In [ ]:
import src.utils
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import seaborn as sns
import sklearn.linear_model

## RNG
rng = np.random.default_rng()

## color palette
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## funcs

In [ ]:
def get_windowed(x):
    return src.utils.get_windowed(x, stride=60, window_size=480)


def get_ddx(x, lons_w=slice(150, 190), lons_e=slice(240, 280)):
    """compute horizontal gradient"""

    ## ranges
    lat_range = dict(latitude=slice(-5, 5))
    # lon_range_w = dict(longitude=slice(120, 160))
    # lon_range_e = dict(longitude=slice(210, 270))
    lon_range_w = dict(longitude=lons_w)
    lon_range_e = dict(longitude=lons_e)

    ## spatial avg
    avg = lambda x: x.mean(["latitude", "longitude"])

    ## get east and west
    Tw = avg(x.sel(**lon_range_w, **lat_range))
    Te = avg(x.sel(**lon_range_e, **lat_range))

    return Te - Tw


def get_T34(x):
    """compute horizontal gradient"""

    ## ranges
    lat_range = dict(latitude=slice(-5, 5))
    lon_range = dict(longitude=slice(190, 240))

    ## spatial avg
    avg = lambda x: x.mean(["latitude", "longitude"])

    ## get east and west
    T34 = avg(x.sel(**lon_range, **lat_range))

    return T34

### Load CESM2 data

In [ ]:
## specify lons for grad
LONS_W = slice(140, 190)
LONS_E = slice(240, 280)
# LONS_W = slice(120,180)
# LONS_E = slice(180,280)

## get_ddx helper
get_ddx_ = lambda x: get_ddx(x, lons_e=LONS_E, lons_w=LONS_W)

## load data
forced, anom = src.utils.load_consolidated()

## subset for temperature
sst = xr.merge([forced[["sst_comp"]], forced["sst"] + anom["sst"]])
ssta = anom[["sst", "sst_comp"]]

## compute grad
dTdx = get_windowed(src.utils.reconstruct_wrapper(sst, get_ddx_)["sst"])
T34 = get_windowed(src.utils.reconstruct_wrapper(sst, get_T34)["sst"])
T34_a = get_windowed(src.utils.reconstruct_wrapper(ssta, get_T34)["sst"])
x = xr.merge([dTdx.rename("dTdx"), T34.rename("T34"), T34_a.rename("T34_a")])

## subset for DJF
x = x.resample({"time": "QS-DEC"}).mean()
x = x.sel(time=slice("1850-12-01", "1888-12-01", 4)).sel(year=1900)
x = x.stack(s=["time", "member"])

### get best fit

In [ ]:
coefs = x["dTdx"].assign_coords({"T": x.T34_a}).polyfit(dim="T", deg=2).to_dataarray()
coefs = coefs.values.flatten()

f = lambda x: coefs[0] * x**2 + coefs[1] * x + coefs[2]
f_sym = lambda x: coefs[1] * x + coefs[2]

Plot nonlinear relation

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3))
ax.scatter(x["T34_a"], x["dTdx"], s=1)
ax.plot(np.linspace(-4, 4), f(np.linspace(-4, 4)), c="k")
ax.plot(np.linspace(-4, 4), f_sym(np.linspace(-4, 4)), c="k", ls="--")

### Stochastic timeseries

#### Generate random data

In [ ]:
## specify trend in sigma (K/century)
sigma_trend = 0.4

## number of years/members for simulation
n_years = 40
n_member = 30000

## generate series of sigma over time
years = np.arange(n_years)
sigma_T34 = 1 + (1e-2 * sigma_trend) * years

## generate random series
T34 = sigma_T34[None, :] * rng.normal(size=(n_member, n_years))

## put in xarray
coords = dict(member=np.arange(n_member), year=years)
T34 = xr.DataArray(T34, coords=coords, dims=["member", "year"])

#### compute trends

In [ ]:
get_trend = (
    lambda z: 100 * z.polyfit(dim="year", deg=1).to_dataarray().sel(degree=1).squeeze()
)
trend_fsym = get_trend(f_sym(T34))
trend_f = get_trend(f(T34))

#### Plot

In [ ]:
edges = np.linspace(-2, 2, 20)
get_pdf = lambda y: src.utils.get_empirical_pdf(y.values, edges=edges)[0]
pdf = get_pdf(trend_fsym)
pdf_f = get_pdf(trend_f)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.stairs(pdf, edges, fill=True, alpha=0.2, color="k", label=r"constant $\sigma$")
ax.stairs(pdf_f, edges, label=r"increasing  $\sigma$")
ax.set_xlim([-1.5, 1.5])
ax.axvline(0, ls="--", c="k", lw=0.8)
ax.legend(prop=dict(size=8))
ax.set_xticks([-1, 0, 1])
ax.set_yticks([])
ax.set_xlabel(r"$dT/dx$ trend (K/century)")
ax.set_ylabel(r"Density")

plt.show()

## Decompose horizontal grad

In [ ]:
## compute grad
dTdx = get_windowed(src.utils.reconstruct_wrapper(sst, get_ddx_)["sst"])
# T34 = get_windowed(src.utils.reconstruct_wrapper(sst, get_T34)["sst"])
T34_a = get_windowed(src.utils.reconstruct_wrapper(ssta, get_T34)["sst"])
x = xr.merge([dTdx.rename("dTdx"), T34_a.rename("T34_a")])

## subset for DJF
x = x.resample({"time": "QS-DEC"}).mean()
x = x.sel(time=slice("1850-12-01", "1888-12-01", 4))

In [ ]:
## compute metrics
sigma_T = x["T34_a"].std("time")
dTdx_ = x["dTdx"].mean("time")
const = xr.ones_like(sigma_T)

## put in xarray
data = xr.merge(
    [sigma_T.rename("sigma_T"), dTdx_.rename("dTdx"), const.rename("const")]
)
data = data.transpose("year", ...)

In [ ]:
## Linear regression object
LR = sklearn.linear_model.LinearRegression

## empty array to hold results
coefs = []

## loop thru years
for year in data.year:

    ## get data
    X = data["sigma_T"].sel(year=year)
    Y = data["dTdx"].sel(year=year)

    ## instantiate model
    mod = LR(fit_intercept=True)
    mod.fit(X=X.values[:, None], y=Y.values)

    coefs.append(np.array([mod.coef_.item(), mod.intercept_]))

## put coefs in array
coefs = np.stack(coefs, axis=0)

## put everything in xarray
data["dTdx_bar"] = coefs[:, 1]
data["dTdx_star"] = coefs[:, :1] * data["sigma_T"]
data["dTdx_star_const"] = coefs[0, 0] * data["sigma_T"]
data["dTdx_bar_res"] = data["dTdx"] - data["dTdx_star"]
data["dTdx_bar_const"] = data["dTdx"] - data["dTdx_star_const"]
data["m"] = coefs[:, 0]

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 2.5), layout="constrained")
# ax.plot(data.year, data["dTdx_bar"])

for ax, v in zip(axs, ["dTdx_bar_res", "dTdx_bar_const"]):

    for q, lw in zip([0.5, 0.05, 0.95], [2, 0.8, 0.8]):
        sel_ = lambda x: x.quantile(q=q, dim="member")

        ## plot corrected
        ax.plot(data.year, sel_(data[v]), lw=lw, c="k")

        ## plot raw
        ax.plot(data.year, sel_(data["dTdx"]), lw=lw, c=sns.color_palette()[0])

    ax.axvline(2010, c="k", ls="--", lw=1)
    ax.set_ylim([None, -3])

plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9, 3), layout="constrained")

for ax, year_i in zip(axs, [6, 16, 26]):

    ## plot data
    ax.scatter(
        data["sigma_T"].isel(year=year_i),
        data["dTdx"].isel(year=year_i),
    )

    ## plot best fit
    zz = np.linspace(1, 2)
    ax.plot(zz, coefs[year_i, 0] * zz + coefs[year_i, 1], c="k", lw=1)
    ax.plot(zz, coefs[6, 0] * zz + coefs[6, 1], c="gray", lw=1)
    ax.set_title(data.year.isel(year=year_i).values.item())
    ax.set_xticks([1.2, 1.5, 1.8])

for ax in axs[1:]:
    ax.set_yticks([])
src.utils.set_lims(axs)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2.5))
# ax.plot(data.year, xr.corr(data["dTdx"], data["sigma_T"], dim="member"))
ax.plot(data.year, data["m"])
ax.axvline(2010)
plt.show()

could do same thing with El Niño/La Niña composite...(instead of regression)